In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.linear_model import LinearRegression


### HAR Modeling

#### AR Models 
Auto Regressive models are used to forecast **Realised variance ($RV$)**. The Simplest one is $AR(1)$ model:
$$ RV_t =\beta_0 +\beta_1 RV_{t-1} $$

and in general $AR(p)$ in a autoregressive model to order $p$. (I am assuming no noise we can add a noise term as well)
$$ RV_T =\beta_0 +\sum_{i=0}^p\beta_i RV_{T-i} .$$

We estimate the $\beta$ parameters using data say for the $0, 1, \dots, T$ to predict the variance (and volatility) at time $T$.

#### HAR Model

Prof Corsi idea that forecasting might actually only depend on the volatility over the last week and month. So essentially HAR is a AR(22) model but with some stronger constarins on the coeeficients. \\
First define the variables, 
$$ RV_{t-1}^w = \frac{1}{5} \sum _{i=1}^5 RV_{t-i}$$

and, 

$$ RV_{t-1}^m = \frac{1}{22} \sum _{i=1}^{22} RV_{t-i}. $$

$RV_{t-1}^w$ and $RV_{t-1}^m$ are the average daily valatility over the last week and month (5 and 22 are the average trading days in a week and month).

The below is a basic implementation where we have calculated the $RV$ using 5 min data from 5 min data over the last 60 days.

In [3]:
ticker = 'TSLA'
stock_data = yf.download(ticker, start = '2026-04-24', end ='2026-06-20', interval = "5m")
log_returns = np.log(stock_data[("Close", ticker)]).diff()
daily_RV = log_returns.groupby(log_returns.index.date).apply(lambda x:(x**2).sum())

har_df = pd.DataFrame({
    "rv_d": daily_RV,
    "rv_w": daily_RV.rolling(5).mean(),
    "rv_m": daily_RV.rolling(22).mean(),
    "target": daily_RV.shift(-1)
}).dropna()

stock_data = stock_data.dropna()

X = har_df[["rv_d", "rv_w", "rv_m"]]
Y = har_df["target"]

model = LinearRegression()
model.fit(X, Y)

print(model.intercept_)
print(model.coef_)


$TSLA: possibly delisted; no price data found  (5m 2026-04-24 -> 2026-06-20) (Yahoo error = "5m data not available for startTime=1777003200 and endTime=1781928000. The requested range must be within the last 60 days.")
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['TSLA']: possibly delisted; no price data found  (5m 2026-04-24 -> 2026-06-20) (Yahoo error = "5m data not available for startTime=1777003200 and endTime=1781928000. The requested range must be within the last 60 days.")


ValueError: Found array with 0 sample(s) (shape=(0, 3)) while a minimum of 1 is required by LinearRegression.

In [ ]:
print(X.corr())

          rv_d      rv_w      rv_m
rv_d  1.000000  0.580234 -0.204318
rv_w  0.580234  1.000000 -0.226722
rv_m -0.204318 -0.226722  1.000000


The issue if yfinance has granuar data like 5 min (anything less than 1 day) only for the last 60 days. The major issues are:
- We were targeting older timeperiods 2020-2022, 2014-2016, so its an issue getting granular data. 
- even using this as an estimate to predict variance and volatility for tommorrow, is not reliable as when fitting the Linear regression will only have like 30 or 40 data points (will have to remove the first 22), so the fit is also not good. 

One alternative is not look for other sources to get more granualar data, a more easier option is to use other indecators of variance like Parkinson, Garman-Klass Volatility analysis. Instead of using 5 min or 15 min data to predict Intraday volatility it uses Open, Close, High, and Low to predict the daily var. So need much less data and very efficient at predicting variance. 

Note (for me): Will try to read up some motivation for where these came from. 

#### Garman-Klass
The (daily) variance using the Garman-Klass method
$$RV (GK) = \frac{1}{2} \ln\left(\frac{H}{L}\right)^2 - (2\ln(2) -1)\ln \left(\frac{C}{O}\right),$$
where, \\
$H, L, C, O$ are the daily High, Low, Close, and Open prices. (The yfinance data contains all for daily data). 

Note: There are slight modifications Garman-Klass-Yang-Zhang where there is a term using the previous day closing as well. ideally to take care of stocks that have after hours trading. 

Note (for me): Will try to read up some motivation for where these came from. 

Below have computed HAR-GK, 

In [ ]:
### Not enough Data so using Open High Low and Close to predict daily RV
## Garman-Klass

start_date = '2020-01-01'
end_date = '2021-01-01'
end_date_test = '2021-02-15'

stock_data = yf.download(ticker, start=start_date, end=end_date)
H = stock_data[("High",ticker)]
L = stock_data[("Low",ticker)]
C = stock_data[("Close",ticker)]
O = stock_data[("Open",ticker)]

## One Suggestion was to use log valatility

daily_RV = 0.5*(np.log(H/L))**2 - (2*np.log(2)-1)*(np.log(C/O))**2

har_df = pd.DataFrame({
    "rv_d": daily_RV,
    "rv_w": daily_RV.rolling(5).mean(),
    "rv_m": daily_RV.rolling(22).mean(),
    "target": daily_RV.shift(-1)
}).dropna()

stock_data = stock_data.dropna()

X = har_df[["rv_d", "rv_w", "rv_m"]]
Y = har_df["target"]

model = LinearRegression()
model.fit(X, Y)

print(model.intercept_)
print(model.coef_)

day1RV = model.predict(X.iloc[[-1]])[0]
vol= np.sqrt(day1RV*252)
print(f"The annualised volatility  for {ticker} on {end_date} is {vol*100}%."  )
model.predict(X.iloc[[-1]])

[*********************100%***********************]  1 of 1 completed

0.0008018311675808073
[ 0.3086265   0.31054833 -0.03346943]
The annualised volatility  for TSLA on 2021-01-01 is 51.09368912497321%.


array([0.00103594])

So I have used the data in 2020 to get parameters for the model, and using it to get the future values below

In [ ]:
## 22 days
history = list(daily_RV.tail(22))
future = []

for _ in range(22):

    rv_d = history[-1]
    rv_w = np.mean(history[-5:])
    rv_m = np.mean(history[-22:])

    X_n = pd.DataFrame({
        "rv_d": [rv_d],
        "rv_w": [rv_w],
        "rv_m": [rv_m]
    })

    pred_rv = model.predict(X_n)[0]

    future.append(pred_rv)
    history.append(pred_rv)

monthly_vol = np.sqrt(np.sum(future) * 252 / 22)

print(f"The monthly Volatility HAR-GK (Annulazed) for Jan is {monthly_vol:0.2%}.")



The monthly Volatility HAR-GK (Annulazed) for Jan is 66.97%.


In [ ]:
## Actual Volatility computation

test_data = yf.download(ticker, start= end_date, end= end_date_test)
close = test_data.iloc[:22][("Close", ticker)]
log_returns = np.log(close/close.shift(1)).dropna()
act_vol= np.std(log_returns)*np.sqrt(252)

print(f"Actual Vol: {act_vol:0.2%}")



[*********************100%***********************]  1 of 1 completed

Actual Vol: 62.12%


In [4]:
## Shall try to plot volatility or something (Not sure in this context what to plot)